# MeshPort · Colab A100

Run all cells, then paste the complete URL printed by the final cell into the localhost app. Upload one image or up to six views of the same object in localhost. Generation runs here; preview and textured FBX conversion run locally.

Select an A100 GPU before running. First setup and downloads can take several minutes. The tunnel requires a random session key included in the URL fragment. Treat the complete URL as private; notebook outputs may contain that key. Clear outputs before sharing the notebook. Keep Colab connected.

Uses TRELLIS-image-large with seed 42, 16 sampling steps and 2048 textures. More runtime does not guarantee better reconstruction. Multi-image conditioning is experimental.


In [ ]:
# @title 0. Verify runtime before setup
import sys, subprocess, pathlib
print("Colab kernel Python:", sys.version)
for cmd in [["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], ["nvcc", "--version"]]:
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout, result.stderr)
print("Existing TRELLIS:", pathlib.Path("/content/TRELLIS").exists())
print("Input images:", [str(p) for p in pathlib.Path("/content").glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}])


In [ ]:
# @title 1. Install an isolated TRELLIS environment (rerunnable)
import os, pathlib, subprocess, textwrap
setup_script = r'''
set -euo pipefail
cd /content
export MAMBA_ROOT_PREFIX=/content/mamba
if [ ! -x /content/bin/micromamba ]; then
  curl -fLs https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
  tar -xjf /content/micromamba.tar.bz2 -C /content bin/micromamba
fi
if [ ! -x /content/trellis-env/bin/python ]; then
  /content/bin/micromamba create -y -p /content/trellis-env -c conda-forge -c nvidia python=3.10 pip cuda-nvcc=12.1 cuda-cudart-dev=12.1 cuda-cccl=12.1 gcc_linux-64=12 gxx_linux-64=12
fi
export PATH=/content/trellis-env/bin:$PATH
export CUDA_HOME=/content/trellis-env
export CC=/content/trellis-env/bin/x86_64-conda-linux-gnu-cc
export CXX=/content/trellis-env/bin/x86_64-conda-linux-gnu-c++
export TORCH_CUDA_ARCH_LIST=8.0 MAX_JOBS=4
export ATTN_BACKEND=xformers SPCONV_ALGO=native
if [ ! -d /content/TRELLIS/.git ]; then
  git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
fi
cd /content/TRELLIS
git rev-parse HEAD
printf 'numpy==1.26.4
scipy==1.14.1
torch==2.4.0
torchvision==0.19.0
xformers==0.0.27.post2
pymeshfix==0.17.0
pyvista==0.44.2
vtk==9.3.1
transformers==4.46.3
huggingface-hub==0.25.2
fastapi==0.115.6
pydantic==2.10.6
starlette==0.41.3
' > /content/trellis-constraints.txt
export PIP_CONSTRAINT=/content/trellis-constraints.txt
python -m pip install --disable-pip-version-check 'setuptools<70' wheel ninja packaging
python -m pip install torch==2.4.0 torchvision==0.19.0 xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
python -m pip install numpy==1.26.4 scipy==1.14.1 pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless rembg==2.0.61 onnxruntime trimesh open3d xatlas pyvista==0.44.2 pymeshfix==0.17.0 igraph transformers==4.46.3 huggingface-hub==0.25.2 spconv-cu120 gradio==4.44.1 fastapi==0.115.6 pydantic==2.10.6 starlette==0.41.3
python -m pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
python -m pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
export CPATH=/content/trellis-env/targets/x86_64-linux/include
for cuda_include in /content/trellis-env/lib/python3.10/site-packages/nvidia/*/include; do
  export CPATH="$CPATH:$cuda_include"
done
export LIBRARY_PATH=/content/trellis-env/targets/x86_64-linux/lib
mkdir -p /content/trellis-extensions
if [ ! -d /content/trellis-extensions/nvdiffrast/.git ]; then
  git clone --branch v0.3.3 https://github.com/NVlabs/nvdiffrast.git /content/trellis-extensions/nvdiffrast
fi
if [ ! -d /content/trellis-extensions/mip-splatting/.git ]; then
  git clone --recursive https://github.com/autonomousvision/mip-splatting.git /content/trellis-extensions/mip-splatting
fi
if ! python -c 'import nvdiffrast' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/nvdiffrast
fi
if ! python -c 'import diff_gaussian_rasterization' >/dev/null 2>&1; then
  python -m pip install --no-build-isolation /content/trellis-extensions/mip-splatting/submodules/diff-gaussian-rasterization
fi
python -m pip check
python -u -c "import torch,numpy,scipy,kaolin,nvdiffrast.torch,diff_gaussian_rasterization; from trellis.pipelines import TrellisImageTo3DPipeline; from trellis.utils import postprocessing_utils; assert torch.cuda.is_available(); nvdiffrast.torch.RasterizeCudaContext(); print('VERIFIED',torch.__version__,numpy.__version__,scipy.__version__,torch.cuda.get_device_name(0))"
printf '
TRELLIS dependency imports passed. Run Cell 2.
'
'''
pathlib.Path('/content/trellis-setup.sh').write_text(textwrap.dedent(setup_script))
with open('/content/trellis-setup.log', 'a') as log:
    proc = subprocess.Popen(['bash', '/content/trellis-setup.sh'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        print(line, end='')
    rc = proc.wait()
if rc:
    raise RuntimeError(f'Setup stopped with exit code {rc}; see /content/trellis-setup.log. Completed packages are reusable.')


In [ ]:
# @title 2. Start the image-to-3D service
import os, pathlib, subprocess, secrets, time, urllib.request
service_source = '"""Authenticated Colab service, embedded into the notebook by build_notebook.py."""\nimport os, io, base64, json, uuid, time, threading, hashlib\nfrom pathlib import Path\nos.environ[\'ATTN_BACKEND\']=\'xformers\'\nos.environ[\'SPCONV_ALGO\']=\'native\'\nimport torch\nfrom PIL import Image\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import FileResponse, JSONResponse\nfrom trellis.pipelines import TrellisImageTo3DPipeline\nfrom trellis.utils import postprocessing_utils\n\nTOKEN=os.environ[\'STUDIO_KEY\']\nROOT=Path(\'/content/trellis-results\'); ROOT.mkdir(exist_ok=True)\npipeline=TrellisImageTo3DPipeline.from_pretrained(\'microsoft/TRELLIS-image-large\')\npipeline.cuda()\napp=FastAPI(docs_url=None,redoc_url=None,openapi_url=None)\njobs={}; lock=threading.Lock()\nImage.MAX_IMAGE_PIXELS=16_000_000\n\n@app.middleware(\'http\')\nasync def auth(request,call_next):\n    import secrets\n    if not secrets.compare_digest(request.headers.get(\'authorization\',\'\'),\'Bearer \'+TOKEN):\n        return JSONResponse({\'detail\':\'Invalid connection key\'},401)\n    return await call_next(request)\n\n@app.get(\'/health\')\ndef health():\n    return {\'protocol\':\'form-studio-v1\',\'ready\':True,\'model\':\'TRELLIS image large\',\'gpu\':torch.cuda.get_device_name(0),\'steps\':16,\'texture_size\':2048}\n\ndef generate(job_id,images):\n    job=jobs[job_id]; start=time.monotonic(); folder=ROOT/job_id\n    try:\n        folder.mkdir()\n        digests=[]\n        for i,image in enumerate(images):\n            path=folder/f\'input-{i+1}.png\';image.save(path);digests.append(hashlib.sha256(path.read_bytes()).hexdigest())\n        job[\'stage\']=\'Generating geometry and appearance\'\n        params=dict(seed=42,formats=[\'gaussian\',\'mesh\'],sparse_structure_sampler_params={\'steps\':16,\'cfg_strength\':7.5},slat_sampler_params={\'steps\':16,\'cfg_strength\':3.0})\n        outputs=pipeline.run(images[0],**params) if len(images)==1 else pipeline.run_multi_image(images,**params)\n        job[\'stage\']=\'Baking 2048 texture\'\n        mesh=postprocessing_utils.to_glb(outputs[\'gaussian\'][0],outputs[\'mesh\'][0],simplify=0.95,texture_size=2048)\n        mesh.export(folder/\'model.glb\')\n        details={\'model\':\'microsoft/TRELLIS-image-large\',\'seed\':42,\'steps\':16,\'texture_size\':2048,\'input_sha256\':digests,\'faces\':len(mesh.faces),\'vertices\':len(mesh.vertices),\'watertight\':bool(mesh.is_watertight),\'seconds\':round(time.monotonic()-start,1)}\n        (folder/\'manifest.json\').write_text(json.dumps(details,indent=2))\n        job.update(status=\'complete\',stage=\'Model ready\',details=details)\n    except Exception as exc:\n        job.update(status=\'failed\',stage=\'Generation failed\',error=str(exc).replace(TOKEN,\'[redacted]\'))\n    finally:\n        job[\'elapsed\']=round(time.monotonic()-start,1)\n        torch.cuda.empty_cache();lock.release()\n\n@app.post(\'/jobs\')\nasync def create(request:Request):\n    if not lock.acquire(blocking=False):raise HTTPException(409,\'GPU is busy. Try again after the current model finishes.\')\n    try:\n        raw=bytearray()\n        async for chunk in request.stream():\n            raw.extend(chunk)\n            if len(raw)>32*1024*1024:raise ValueError(\'Images exceed 24 MB total upload limit\')\n        body=json.loads(raw); encoded=body[\'images\']\n        if not isinstance(encoded,list) or not 1<=len(encoded)<=6:raise ValueError(\'Choose one to six views of one object\')\n        images=[]\n        for item in encoded:\n            image=Image.open(io.BytesIO(base64.b64decode(item,validate=True)))\n            if image.width*image.height>16_000_000:raise ValueError(\'Each image must be at most 16 megapixels\')\n            if image.format not in {\'PNG\',\'JPEG\',\'WEBP\'}:raise ValueError(\'Use PNG, JPEG or WebP\')\n            image.load();images.append(image.convert(\'RGBA\'))\n        job_id=uuid.uuid4().hex\n        jobs[job_id]={\'id\':job_id,\'status\':\'running\',\'stage\':\'Preparing images\',\'elapsed\':0}\n        threading.Thread(target=generate,args=(job_id,images),daemon=True).start()\n        return {\'id\':job_id}\n    except Exception as exc:\n        lock.release();raise HTTPException(400,str(exc))\n\n@app.get(\'/jobs/{job_id}\')\ndef status(job_id:str):\n    if job_id not in jobs:raise HTTPException(404)\n    return jobs[job_id]\n\n@app.get(\'/jobs/{job_id}/model\')\ndef model(job_id:str):\n    if job_id not in jobs or jobs[job_id][\'status\']!=\'complete\':raise HTTPException(404)\n    return FileResponse(ROOT/job_id/\'model.glb\',media_type=\'model/gltf-binary\')\n\nif __name__==\'__main__\':\n    import uvicorn\n    uvicorn.run(app,host=\'127.0.0.1\',port=7860,access_log=False)\n'

if 'studio_proc' in globals() and studio_proc.poll() is None:
    raise RuntimeError('The service is already running. Run the final cell to copy its URL, or restart the runtime for changed code.')
STUDIO_KEY=secrets.token_urlsafe(32)
path=pathlib.Path('/content/TRELLIS/form_service.py')
path.write_text(service_source)
env=os.environ.copy()
env.update(STUDIO_KEY=STUDIO_KEY, PATH='/content/trellis-env/bin:'+env['PATH'], CUDA_HOME='/content/trellis-env', CC='/content/trellis-env/bin/x86_64-conda-linux-gnu-cc', CXX='/content/trellis-env/bin/x86_64-conda-linux-gnu-c++', ATTN_BACKEND='xformers', SPCONV_ALGO='native', TORCH_CUDA_ARCH_LIST='8.0', MAX_JOBS='4')
env['CPATH']=':'.join(['/content/trellis-env/targets/x86_64-linux/include']+[str(p) for p in pathlib.Path('/content/trellis-env/lib/python3.10/site-packages/nvidia').glob('*/include')])
env['LIBRARY_PATH']='/content/trellis-env/targets/x86_64-linux/lib'
log=open('/content/form-service.log','w')
studio_proc=subprocess.Popen(['/content/trellis-env/bin/python','-u',str(path)],cwd='/content/TRELLIS',env=env,stdout=log,stderr=subprocess.STDOUT)
for attempt in range(600):
    if studio_proc.poll() is not None:
        raise RuntimeError(pathlib.Path('/content/form-service.log').read_text()[-6000:])
    try:
        request=urllib.request.Request('http://127.0.0.1:7860/health',headers={'Authorization':'Bearer '+STUDIO_KEY})
        with urllib.request.urlopen(request,timeout=2) as response:
            if response.status==200:break
    except Exception:time.sleep(2)
else:raise TimeoutError('Loading is taking longer than expected. Inspect /content/form-service.log before restarting.')
print('TRELLIS ready. Run the final cell to get your connection URL.')


In [ ]:
# @title 3. Copy this connection URL into localhost
import pathlib, subprocess, urllib.request, re, time
if 'studio_proc' not in globals() or studio_proc.poll() is not None:
    raise RuntimeError('Run the service cell first.')
if 'tunnel_proc' not in globals() or tunnel_proc.poll() is not None:
    binary=pathlib.Path('/content/cloudflared')
    if not binary.exists():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binary)
        binary.chmod(0o755)
    tunnel_log=open('/content/form-tunnel.log','w')
    tunnel_proc=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
for attempt in range(90):
    if tunnel_proc.poll() is not None:raise RuntimeError('Tunnel stopped; inspect /content/form-tunnel.log')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',pathlib.Path('/content/form-tunnel.log').read_text())
    if match:break
    time.sleep(1)
else:raise TimeoutError('Tunnel URL not ready; rerun this cell.')
print('Paste this entire URL into MeshPort (keep it private):')
print(match.group(0)+'/#key='+STUDIO_KEY)
print('Keep this Colab runtime connected. Reconnecting a new runtime creates a new URL.')
